## OpenAI 연동 기본 예제

- `ChatOpenAI`로 OpenAI API에 연결하고, Prompt → LLM → OutputParser를 LCEL로 연결합니다.
- 사전 준비: `.env` 파일에 `OPENAI_API_KEY` 설정, `uv add langchain-openai python-dotenv`

In [1]:
import os
from dotenv import load_dotenv

#load_dotenv(dotenv_path='.env')
load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

# 키 값은 출력하지 않고 설정 여부만 확인
if not OPENAI_API_KEY:
    raise RuntimeError(".env 파일에 OPENAI_API_KEY 가 없습니다.")
print("OPENAI_API_KEY 설정 확인 완료")

OPENAI_API_KEY 설정 확인 완료


#### OpenAI API로 호출 가능한 모델 목록 확인

- `openai` SDK(`langchain-openai`가 내부적으로 사용)의 `client.models.list()`로 현재 API 키로 사용할 수 있는 모델을 조회합니다.
- 목록 조회는 토큰을 사용하지 않아 비용이 발생하지 않습니다.
- 계정/조직에 따라 목록이 다를 수 있으며, 채팅 모델 외에 임베딩, 이미지, 음성 모델도 함께 나옵니다.

In [2]:
# openai SDK 불러오기 (langchain-openai가 내부적으로 사용하는 SDK)
from openai import OpenAI

# API 키로 클라이언트 생성 (OPENAI_API_KEY는 load_dotenv()로 .env에서 읽어 둔 값)
client = OpenAI(api_key=OPENAI_API_KEY)

# 호출 가능한 모델 목록 조회 (id 기준 오름차순 정렬)
models = sorted(client.models.list().data, key=lambda m: m.id)
print(f"조회된 모델 수: {len(models)}\n")
for m in models:
    print(f"{m.id:<45} {m.owned_by}")

조회된 모델 수: 135

babbage-002                                   system
chat-latest                                   system
chatgpt-image-latest                          system
davinci-002                                   system
gpt-3.5-turbo                                 openai
gpt-3.5-turbo-0125                            system
gpt-3.5-turbo-1106                            system
gpt-3.5-turbo-16k                             openai-internal
gpt-3.5-turbo-instruct                        system
gpt-3.5-turbo-instruct-0914                   system
gpt-4                                         openai
gpt-4-0613                                    openai
gpt-4-turbo                                   system
gpt-4-turbo-2024-04-09                        system
gpt-4.1                                       system
gpt-4.1-2025-04-14                            system
gpt-4.1-mini                                  system
gpt-4.1-mini-2025-04-14                       system
gpt-4.1-nano          

In [3]:
from langchain_core.prompts import ChatPromptTemplate

# prompt
prompt = ChatPromptTemplate.from_messages(
    [ ("system", "당신은 개발자입니다.") , 
     ("user", "{input}") ]
)
print(type(prompt))
print(prompt)

<class 'langchain_core.prompts.chat.ChatPromptTemplate'>
input_variables=['input'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 개발자입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})]


In [4]:
prompt_text = prompt.format(input="파이썬은 무엇인가요? 자세하게 설명해주세요")
print(type(prompt_text))
print(prompt_text)

<class 'str'>
System: 당신은 개발자입니다.
Human: 파이썬은 무엇인가요? 자세하게 설명해주세요


In [5]:
from langchain_openai import ChatOpenAI

# 모델명은 상수로 분리
OPENAI_MODEL = "gpt-4o-mini"

# OpenAI API를 사용하는 ChatOpenAI 인스턴스 생성 (base_url 생략 시 OpenAI 기본 엔드포인트 사용)
llm = ChatOpenAI(
    api_key=OPENAI_API_KEY,
    model=OPENAI_MODEL,
    temperature=0.7
)
print(type(llm))
print(llm.model_name)

<class 'langchain_openai.chat_models.base.ChatOpenAI'>
gpt-4o-mini


In [6]:
try:
    response = llm.invoke(prompt_text)
    print(type(response))
    print("응답:", response.content)
except Exception as e:
    print(f"오류 발생: {e}")

<class 'langchain_core.messages.ai.AIMessage'>
응답: 파이썬(Python)은 높은 수준의 프로그래밍 언어로, 1991년 귀도 반 로썸(Guido van Rossum)에 의해 처음 발표되었습니다. 파이썬은 코드의 가독성이 뛰어나고, 문법이 간단하여 배우기 쉽고, 다양한 프로그래밍 패러다임을 지원합니다. 다음은 파이썬의 주요 특징과 장점입니다.

### 1. **간결하고 명확한 문법**
파이썬은 코드가 읽기 쉽고 쓰기 쉬운 구조를 가지고 있습니다. 이는 개발자가 다른 사람이 작성한 코드를 쉽게 이해할 수 있게 하여 협업에 유리합니다.

### 2. **다양한 프로그래밍 패러다임 지원**
파이썬은 절차적 프로그래밍, 객체지향 프로그래밍, 그리고 함수형 프로그래밍을 지원합니다. 이로 인해 개발자는 다양한 방법으로 문제를 해결할 수 있습니다.

### 3. **광범위한 라이브러리와 프레임워크**
파이썬은 데이터 과학, 웹 개발, 자동화, 인공지능, 머신러닝 등 다양한 분야에서 활용할 수 있는 방대한 라이브러리와 프레임워크를 제공합니다. 예를 들어:
- **웹 개발**: Django, Flask
- **데이터 분석**: Pandas, NumPy
- **과학 계산**: SciPy
- **머신러닝**: TensorFlow, scikit-learn

### 4. **크로스 플랫폼**
파이썬은 Windows, macOS, Linux 등 다양한 운영체제에서 실행할 수 있어 호환성이 뛰어납니다.

### 5. **활발한 커뮤니티**
파이썬은 전 세계적으로 큰 커뮤니티를 가지고 있으며, 많은 자료와 튜토리얼, 포럼이 존재합니다. 이로 인해 문제 해결이나 새로운 기술을 배우기가 용이합니다.

### 6. **인터프리터 언어**
파이썬은 인터프리터 언어로, 코드를 작성한 후 바로 실행할 수 있습니다. 이로 인해 개발 과정에서 빠른 피드백을 받을 수 있습니다.

### 7. **동적 타이핑**
파이썬은 동적 타이핑 언어로, 변수의 타입을 명시적으로 지

## LCEL(LangChain Expression Language)
### Prompt + LLM 모델 + OutputParser

In [7]:
from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()

In [8]:
chain = prompt | llm | output_parser
print(type(chain))
print(chain)

<class 'langchain_core.runnables.base.RunnableSequence'>
first=ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='당신은 개발자입니다.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})]) middle=[ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.6.7', 'langchain': '1.4.3', 'langchain-openai': '1.6.7'}}, profile={'name': 'GPT-4o mini', 'release_date': '2024-07-18', 'last_updated': '2024-07-18', 'open_weights': False, 'max_input_tokens': 128000, 'max_output_tokens': 16384, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 't

In [9]:
response = chain.invoke({"input":"LangChain은 무엇인가요?"})

In [10]:
print(type(response))
print(response)

<class 'langchain_core.messages.base.TextAccessor'>
LangChain은 자연어 처리(NLP) 작업을 위한 프레임워크로, 대화형 애플리케이션이나 다양한 언어 모델을 활용한 응용 프로그램을 개발하는 데 유용합니다. 이 프레임워크는 언어 모델을 효과적으로 사용하고, 데이터 소스와의 상호작용, 사용자 입력 처리, 다양한 작업을 수행할 수 있도록 구성된 여러 가지 모듈을 제공합니다.

LangChain의 주요 특징은 다음과 같습니다:

1. **모듈화**: LangChain은 다양한 기능을 모듈화하여 제공하므로, 필요에 따라 특정 모듈만 선택하여 사용할 수 있습니다.

2. **언어 모델 통합**: OpenAI의 GPT와 같은 다양한 언어 모델과 쉽게 통합할 수 있어, 사용자가 원하는 기능을 손쉽게 구현할 수 있습니다.

3. **데이터 소스 연결**: 데이터베이스, API 등 다양한 데이터 소스와 연결하여, 언어 모델이 실시간 데이터에 접근하고 이를 활용할 수 있도록 지원합니다.

4. **프롬프트 관리**: 사용자 입력을 효과적으로 처리하고, 프롬프트를 동적으로 생성하거나 수정할 수 있는 기능을 제공합니다.

5. **작업 체계**: 체계적이고 유연한 작업 흐름을 구성할 수 있어, 복잡한 대화형 애플리케이션이나 NLP 솔루션을 쉽게 구축할 수 있습니다.

LangChain은 AI 기반의 애플리케이션을 개발하고자 하는 개발자들에게 매우 유용한 도구가 될 수 있습니다.
